# XN Benchmark: PDF / Image → SMILES on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imabhi29/xn-pdf2smiles-benchmark/blob/main/notebooks/run_benchmark_colab.ipynb)

Runs the whole benchmark from GitHub: clone → install → test → download data → run → view and save results.

**Stack:** PyMuPDF (text layer) · DECIMER Segmentation (detection/cropping) · MolScribe (image → SMILES) · RDKit (validation) · PubChem/ChEMBL (verification) · RDKit + pandas (evaluation)

**How to use:** optionally pick *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. No restart is needed.

**Why a separate Python?** MolScribe's dependencies only install on Python 3.10, while Colab ships a newer Python. Step 2 therefore builds a private Python 3.10 environment (`.venv`) and every benchmark command runs with it, so results are the same on any Colab account.

## Step 1 · Settings

In [ ]:
# ---- Settings -------------------------------------------------------------
REPO_URL = "https://github.com/imabhi29/xn-pdf2smiles-benchmark.git"
BRANCH = "main"

MODE = "images"        # "images" = BioVista OCSR benchmark, "pdfs" = full PDF pipeline
LIMIT = 50             # images/PDFs to process; None = config default (500-image sample)
VERIFY = True          # PubChem lookups (adds ~0.3 s per molecule)
SAVE_TO_DRIVE = False  # True = keep results in Google Drive after the session ends
# ---------------------------------------------------------------------------

import os
REPO_DIR = "/content/xn-pdf2smiles-benchmark"
VENV_PY = f"{REPO_DIR}/.venv/bin/python"          # Python 3.10 environment built in Step 2
RUN_NAME = f"{MODE}_{LIMIT or 'default'}"
OUTPUT_DIR = "/content/drive/MyDrive/xn-benchmark-results" if SAVE_TO_DRIVE else f"{REPO_DIR}/results"
RUN_DIR = f"{OUTPUT_DIR}/{RUN_NAME}"
print("Run folder will be:", RUN_DIR)

## Step 2 · Clone the repository and build the environment
About 5–8 minutes the first time (it downloads PyTorch and TensorFlow). Re-running is instant.

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "pull", "-q"], check=False)

!bash scripts/setup_colab.sh
!pip install -q rdkit   # for structure drawings in this notebook's results cells

## Step 3 · Check the install
Unit tests for metrics, chemistry and data loading.

In [ ]:
%cd {REPO_DIR}
!{VENV_PY} -m pytest -q

## Step 4 · Download public data
BioVista OCSR images + labels (Google Drive) and two open-access example PDFs. Skips anything already downloaded.

If Google Drive reports a quota error, wait a few minutes and retry, or download the file manually from the link in `configs/default.yaml` and unzip it into `data/biovista_ocsr/`.

In [ ]:
%cd {REPO_DIR}
import subprocess
result = subprocess.run([VENV_PY, "scripts/download_data.py", "--what", "all"])
if result.returncode != 0:
    raise RuntimeError("Data download failed - see the messages above. If it is a Google Drive quota "
                       "error, wait a few minutes and re-run this cell.")

## Step 5 · Run the benchmark

In [ ]:
%cd {REPO_DIR}
import subprocess, sys
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU: running on CPU (slower)"
cmd = [VENV_PY, "-m", "xnbench.run", "--mode", MODE, "--run-name", RUN_NAME, "--output-dir", OUTPUT_DIR]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]
if not VERIFY:
    cmd += ["--no-verify"]
print(" ".join(cmd), "\n")

# Stream the output live, and stop "Run all" here if the benchmark fails
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
lines = []
for line in proc.stdout:
    print(line, end="")
    lines.append(line)
if proc.wait() != 0:
    raise RuntimeError("Benchmark failed. The error is in the last lines above:\n\n" + "".join(lines[-15:]))

## Step 6 · Results

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if not os.path.exists(f"{RUN_DIR}/metrics.json"):
    raise FileNotFoundError(f"No results in {RUN_DIR}. Step 5 did not finish - check its output.")

metrics = json.load(open(f"{RUN_DIR}/metrics.json"))
print(json.dumps(metrics, indent=2))

if MODE == "images":
    by_cat = pd.read_csv(f"{RUN_DIR}/metrics_by_category.csv")
    display(by_cat)
    cols = ["validity_rate", "exact_match_rate", "connectivity_match_rate", "mean_tanimoto"]
    by_cat.set_index("category")[cols].plot.bar(figsize=(8, 4), ylim=(0, 1), rot=0)
    plt.title("MolScribe on BioVista OCSR")
    plt.ylabel("score")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    for name in ["metrics_by_pdf.csv", "missed_molecules.csv"]:
        path = f"{RUN_DIR}/{name}"
        if os.path.exists(path):
            print(name); display(pd.read_csv(path).head(20))

### Error gallery
Wrong predictions side by side: input image, ground-truth structure, predicted structure.

In [ ]:
from PIL import Image
from rdkit import Chem
from rdkit.Chem import Draw

preds = pd.read_csv(f"{RUN_DIR}/predictions.csv")
if MODE == "images":
    wrong = preds[~preds["exact_match"].astype(bool)].head(6)
    for _, r in wrong.iterrows():
        fig, ax = plt.subplots(1, 3, figsize=(11, 3))
        ax[0].imshow(Image.open(r["image_path"])); ax[0].set_title("input image")
        for a, smi, title in [(ax[1], r["true_smiles"], "ground truth"),
                              (ax[2], r["pred_smiles"], f"predicted (Tanimoto={r['tanimoto']})")]:
            mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
            if mol is not None:
                a.imshow(Draw.MolToImage(mol, size=(300, 300)))
            else:
                a.text(0.5, 0.5, "invalid SMILES", ha="center")
            a.set_title(title, fontsize=9)
        for a in ax: a.axis("off")
        plt.show()
else:
    display(preds[["pdf", "page", "label", "pred_smiles", "pred_valid", "confidence"]].head(20))

## Step 7 · Save results
Downloads a zip of the run folder. (With `SAVE_TO_DRIVE = True`, results are already in Google Drive.)

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive(f"/content/{RUN_NAME}", "zip", RUN_DIR)
files.download(archive)